In [2]:
! pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]


In [5]:
#!/usr/bin/env python3
"""Create an Excel matrix showing video availability for Day 1 through Day 75.

Expected directory structure:

    BASE_DIR/
        user name/
            mp4/
                day1.mp4
                day_02_part1.mp4
                ...

The scanner also recognizes purely numeric names such as ``3.mp4`` and nested
folders such as ``mp4/day4/clip.mp4``. It intentionally does not treat every
number in a filename as a day, which avoids mistaking dates or video IDs for
day numbers.
"""

from __future__ import annotations

import argparse
import csv
import re
import sys
from pathlib import Path


DEFAULT_BASE_DIR = Path("/mnt/storage/data/youtube/updated_75users/daily")
DEFAULT_OUTPUT = Path("video_day_availability.xlsx")
TICK = "✓"

# Explicit forms: day1, Day 01, day-1, day_001, and variants with extra text.
EXPLICIT_DAY_RE = re.compile(r"(?i)(?<![a-z0-9])day[\s_.-]*0*(\d{1,3})(?!\d)")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Mark which users have videos for Day 1 through Day 75."
    )
    parser.add_argument(
        "base_dir",
        nargs="?",
        type=Path,
        default=DEFAULT_BASE_DIR,
        help=f"Folder containing one directory per user (default: {DEFAULT_BASE_DIR})",
    )
    parser.add_argument(
        "-o",
        "--output",
        type=Path,
        default=DEFAULT_OUTPUT,
        help=f"Output .xlsx or .csv file (default: {DEFAULT_OUTPUT})",
    )
    parser.add_argument(
        "--days",
        type=int,
        default=75,
        help="Number of day columns to create (default: 75)",
    )
    parser.add_argument(
        "--video-folder",
        default="mp4",
        help="Video subfolder inside each user folder (default: mp4)",
    )
    # Jupyter/IPython adds this automatically when code is run through a kernel.
    # Accept and ignore it so the same script works in notebooks and terminals.
    parser.add_argument("-f", "--f", dest="jupyter_kernel_file", help=argparse.SUPPRESS)
    return parser.parse_args()


def day_from_video_path(video_path: Path, video_root: Path, max_day: int) -> int | None:
    """Return the day represented by a video path, or None if not recognized."""
    relative = video_path.relative_to(video_root)

    # Search every component below mp4. This supports mp4/day1/clip.mp4.
    for component in relative.parts:
        text = Path(component).stem if component == relative.name else component
        match = EXPLICIT_DAY_RE.search(text)
        if match:
            day = int(match.group(1))
            return day if 1 <= day <= max_day else None

    # Also support a purely numeric filename such as 01.mp4.
    if video_path.stem.isdigit():
        day = int(video_path.stem)
        return day if 1 <= day <= max_day else None

    return None


def find_days(video_root: Path, max_day: int) -> set[int]:
    days: set[int] = set()
    if not video_root.is_dir():
        return days

    for video_path in video_root.rglob("*"):
        if video_path.is_file() and video_path.suffix.lower() == ".mp4":
            day = day_from_video_path(video_path, video_root, max_day)
            if day is not None:
                days.add(day)
    return days


def collect_users(base_dir: Path, video_folder: str, max_day: int) -> list[tuple[str, set[int]]]:
    users: list[tuple[str, set[int]]] = []
    for user_dir in sorted(
        (path for path in base_dir.iterdir() if path.is_dir() and not path.name.startswith(".")),
        key=lambda path: path.name.casefold(),
    ):
        users.append((user_dir.name, find_days(user_dir / video_folder, max_day)))
    return users


def rows_for(users: list[tuple[str, set[int]]], max_day: int) -> list[list[str]]:
    rows = [["User"] + [f"Day {day}" for day in range(1, max_day + 1)]]
    rows.extend(
        [user_name] + [TICK if day in existing_days else "" for day in range(1, max_day + 1)]
        for user_name, existing_days in users
    )
    return rows


def create_csv(users: list[tuple[str, set[int]]], output: Path, max_day: int) -> None:
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open("w", newline="", encoding="utf-8-sig") as file:
        csv.writer(file).writerows(rows_for(users, max_day))


def create_workbook(users: list[tuple[str, set[int]]], output: Path, max_day: int) -> None:
    try:
        from openpyxl import Workbook
        from openpyxl.formatting.rule import FormulaRule
        from openpyxl.styles import Alignment, Font, PatternFill
        from openpyxl.utils import get_column_letter
    except ImportError:  # pragma: no cover - depends on the user's environment
        print(
            "Creating .xlsx files requires openpyxl. Install it with: pip install openpyxl\n"
            "Or choose a dependency-free CSV output, for example: -o report.csv",
            file=sys.stderr,
        )
        raise SystemExit(2)

    workbook = Workbook()
    sheet = workbook.active
    sheet.title = "Video days"

    for row in rows_for(users, max_day):
        sheet.append(row)

    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(color="FFFFFF", bold=True)
    tick_fill = PatternFill("solid", fgColor="E2F0D9")

    for cell in sheet[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")

    sheet.freeze_panes = "B2"
    sheet.auto_filter.ref = sheet.dimensions
    sheet.row_dimensions[1].height = 24
    sheet.column_dimensions["A"].width = 28

    for column_number in range(2, max_day + 2):
        column_letter = get_column_letter(column_number)
        sheet.column_dimensions[column_letter].width = 7
        for cell in sheet[column_letter][1:]:
            cell.alignment = Alignment(horizontal="center", vertical="center")

    if users:
        final_cell = f"{get_column_letter(max_day + 1)}{len(users) + 1}"
        sheet.conditional_formatting.add(
            f"B2:{final_cell}",
            FormulaRule(formula=['B2="✓"'], fill=tick_fill),
        )

    sheet.sheet_view.showGridLines = False
    output.parent.mkdir(parents=True, exist_ok=True)
    workbook.save(output)


def main() -> int:
    args = parse_args()

    if args.days < 1:
        print("--days must be at least 1", file=sys.stderr)
        return 2
    if not args.base_dir.is_dir():
        print(f"Base directory not found: {args.base_dir}", file=sys.stderr)
        return 1
    output_suffix = args.output.suffix.lower()
    if output_suffix not in {".xlsx", ".csv"}:
        print("Output filename must end in .xlsx or .csv", file=sys.stderr)
        return 2

    users = collect_users(args.base_dir, args.video_folder, args.days)
    if output_suffix == ".csv":
        create_csv(users, args.output, args.days)
    else:
        create_workbook(users, args.output, args.days)

    users_with_missing_days = sum(len(days) < args.days for _, days in users)
    print(f"Created: {args.output.resolve()}")
    print(f"Users scanned: {len(users)}")
    print(f"Users with at least one missing day: {users_with_missing_days}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Created: /home/ganand/Documents/Youtube-Persistence/Code/Data Analysis/video_day_availability.xlsx
Users scanned: 70
Users with at least one missing day: 70


SystemExit: 0

/home/ganand/miniconda3/envs/yp/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
